In [1]:
import os, io
import sys
import json
import boto3
import mlflow
import polars as pl
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

/mnt/e/Repos/Projects/ML-Model-Factory/.venv/lib/python3.10/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [2]:
load_dotenv()
# MinIO (Data Storage)

MINIO_ENDPOINT = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
MINIO_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
MINIO_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
DATA_BUCKET = "processed-features"
TEST_DATA_KEY = "test_data.parquet" 

In [3]:
# MLflow (Model Registry)
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = "mobile_sales_prediction"
MODEL_NAME = "mobile-sales-predictor"
ALIAS_CHAMPION = "champion"
TARGET_COL = "Quantity Sold"
R2_THRESHOLD = 0.01

In [4]:
s3 = boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY
    )

In [5]:
obj = s3.get_object(Bucket=DATA_BUCKET, Key=TEST_DATA_KEY)
data = io.BytesIO(obj['Body'].read())
df = pl.read_parquet(data)

In [6]:
df.columns

['Price',
 'days_to_sell',
 'dispatch_year',
 'dispatch_month',
 'dispatch_day_of_week',
 'spec_length',
 'brand_code',
 'region_code',
 'ram_code',
 'rom_code',
 'avg_price_per_brand',
 'avg_qty_per_region',
 'Quantity Sold']

In [7]:
df.shape

(4997, 13)

In [8]:
X_test = df.drop(TARGET_COL).to_pandas()
y_test = df[TARGET_COL].to_pandas()
X_test.columns

Index(['Price', 'days_to_sell', 'dispatch_year', 'dispatch_month',
       'dispatch_day_of_week', 'spec_length', 'brand_code', 'region_code',
       'ram_code', 'rom_code', 'avg_price_per_brand', 'avg_qty_per_region'],
      dtype='object')

In [9]:
def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = root_mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}

In [10]:
mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.tracking.MlflowClient()

In [11]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

In [12]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=1
)
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.mae,metrics.mse,metrics.r2,params.features,params.test_size,params.model_type,params.n_estimators,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.user,tags.mlflow.log-model.history,tags.mlflow.source.name
0,757d23c7fadf454da35e7fc8f9640025,1,FINISHED,mlflow-artifacts:/1/757d23c7fadf454da35e7fc8f9...,2026-09-08 16:32:33.772000+00:00,2026-09-08 16:32:49.539000+00:00,2.507376,8.449368,-0.027241,"['Price', 'days_to_sell', 'dispatch_year', 'di...",0.2,RandomForestRegressor,100,rebellious-fish-37,LOCAL,root,"[{""run_id"": ""757d23c7fadf454da35e7fc8f9640025""...",train.py


In [13]:
RUN_ID = runs.iloc[0]["run_id"]
RUN_ID

'757d23c7fadf454da35e7fc8f9640025'

In [14]:
new_model = mlflow.sklearn.load_model(f"runs:/{RUN_ID}/model")

/mnt/e/Repos/Projects/ML-Model-Factory/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.4.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/e/Repos/Projects/ML-Model-Factory/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.4.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
new_metrics = evaluate_model(new_model, X_test, y_test)

In [16]:
new_metrics

{'mae': 2.5073764258555133,
 'rmse': 2.9067796236846717,
 'r2': -0.02724084476410793}

In [17]:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
versions

[<ModelVersion: aliases=[], creation_timestamp=1788885169475, current_stage='None', description='', last_updated_timestamp=1788885169475, name='mobile-sales-predictor', run_id='757d23c7fadf454da35e7fc8f9640025', run_link='', source='mlflow-artifacts:/1/757d23c7fadf454da35e7fc8f9640025/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='1'>]

In [ ]:
for v in versions:
    if v.run_id == RUN_ID:
        target_version = v.version
target_version

'1'

In [ ]:
# Promote first model to champion if no champion exists
client.set_registered_model_alias(MODEL_NAME, ALIAS_CHAMPION, target_version)